# Suggested walks pipeline: street network to walks

This notebook is where the CatWalk's **walks** come from: loops of about 2, 3 and 5 km that start and end at the same spot and pass cats, green alleys, community gardens and water fountains on the way.

It does two jobs:

1. **Steps 1–7: the original Python version of the walks.** It plans 18 loops (3 starting points in the Plateau × 2, 3 and 5 km × two themes) and saves them to `data/walks.geojson`:
   - 🐈 **Cat walk**: passes as many of my cat sightings as possible.
   - 🌿 **Nature walk**: uses as much of the green alleys as possible, and passes community gardens where it can.
2. **Step 8: exports the walking network** to `data/walk_network.json`, a small file the website downloads so it can plan walks itself, from any spot, in the reader's browser.

```
OpenStreetMap (downloaded by OSMnx)  -> data/plateau_walk.graphml   the walking network, saved so it downloads once
data/cat_photos.geojson, ruelles_vertes.geojson, communal_gardens.geojson, water_fountains.geojson
                                     -> data/walks.geojson          18 ready-made walks (steps 4-7)
plateau_walk.graphml + green alleys  -> data/walk_network.json      the network for the website (step 8)
```

**What the website uses today:** only `walk_network.json` (step 8). Since the "Stops to include" update, every walk on the map, including the three fixed starts, is planned in the browser by `app.js`, and the map no longer loads `walks.geojson`. Steps 4–7 are kept because they are where the method was first worked out and tested, and they are still the easiest place to read how a loop is made: `app.js` uses the same ideas (see "How I got here" below).

Run each cell in order, top to bottom. The whole notebook takes about a minute, plus about a minute more the first time to download the network.

## How I got here

The walks were built in stages, each one coming from a test or a need I ran into. (I built them with a lot of help from Claude; see the AI disclosure in the README.)

1. **The A3 idea.** For Assignment 3 I wanted readers to be able to *build a cat walk*: choose a start and a length (2, 3 or 5 km) and get a loop that visits as many cats as possible, or one that focuses on nature (green alleys and gardens). The README's A3 section has the plan and the tests.

2. **First version: ready-made walks from Python (steps 1–7).** Finding a route needs the whole street network and a route-finding algorithm. Python has both ready to use (**OSMnx** downloads OpenStreetMap streets, **NetworkX** finds shortest routes), so the first version planned every walk *in advance* in this notebook. I picked three starts that are easy to reach without a car (Métro Mont-Royal, Métro Laurier, Parc Laurier), and the notebook saved the 3 × 3 × 2 = 18 best loops to `walks.geojson`. The map only had to draw the one matching the reader's menus (commit "Add suggested walks box to the map and record A3 test results").

3. **The edge-case test needed a typed start: "My own spot".** My Test 3 was to type a start outside the network (like "Ottawa") and check for a "sorry" message, but with only a menu of three starts there was nothing to type. So I added **"My own spot…"** (an address or a tap on the map). Walks from any spot can't be prepared in advance, and Python doesn't run on a website, so **step 8** was added: it exports the walking network as a small JSON file, and `app.js` plans the loop in the browser with the same method as step 5 (commit "added own walking routes").

4. **Themes became a checklist.** The cat/nature themes were replaced by **"Stops to include"** (Cats, Water fountains, Nature, Free AC, and later "My cats"). With any mix of stops possible, ready-made walks no longer made sense, so now *every* walk is planned in the browser from `walk_network.json`. The planner in `app.js` (`planLoop`) grew from the method here: instead of always two corners, it picks one corner next to each ticked kind of stop (plus 0–2 extra "ring" corners to stretch the loop to length), visits them in compass order around the start so the loop doesn't zigzag, tries 160 loops instead of 300 (so it's fast enough in a browser), and keeps the loop that passes the most ticked kinds.

So the **ideas** in step 5 (ring, random tries, filter, score) are still exactly how the map's walks are made, just written in JavaScript.

## The key ideas, in plain words

- **The street network is a graph.** Every street corner (or dead end) is a *node*, and every piece of street, path or alley between two corners is an *edge* with its length in metres. Once the streets are a graph, "find a walking route" becomes "find a path through the graph", which computers are very good at.
- **Shortest route = Dijkstra's algorithm.** `nx.shortest_path(..., weight="length")` uses Dijkstra's algorithm: starting from one corner, it spreads outward, always extending the closest unfinished corner first, until it reaches the destination. The result is guaranteed to be the shortest route.
- **The weight decides what "shortest" means.** Usually it's metres. For nature walks, green alley pieces are given a *lower* cost than their real length (step 3), so the "shortest" route bends towards green alleys. Nothing else in the route-finding has to change.
- **There's no ready-made "give me a nice 3 km loop" function.** Shortest-route tools find the *shortest* way from A to B, but a loop goes from A back to A, and we want it to be a set length. Checking every possible loop would take forever (the number of possible routes in 13,000 corners is astronomical). So the notebook uses **generate and test**: build many candidate loops quickly, throw out the bad ones, keep the best. It won't always find the perfect loop, but it reliably finds a very good one in about a second.
- **The triangle trick for the right length.** A loop *start → a → b → start* made of three shortest routes is roughly a triangle. If corners `a` and `b` are each about **a third of the walk's length** away from the start (the "ring"), the two legs to and from the start add up to about two thirds of the target, and the leg between `a` and `b` adds roughly the rest. It's only approximate, which is why loops that come out too short or too long are thrown away afterwards.
- **Scoring picks the winner.** Each valid loop gets a number (how many cats, how many metres of green alley, how many gardens), and the highest number wins.
- **Metres, not degrees.** Longitude and latitude are angles, and a degree of longitude isn't the same distance as a degree of latitude. All measuring (lengths, "within 50 m", "8 m of slack") is done in a projection measured in metres, then turned back into longitude and latitude for the website.

## 0) One-time setup
**OSMnx** downloads street networks from OpenStreetMap and finds routes on them (it is built on **NetworkX**, the general Python library for graphs). It needs **SciPy** to find the nearest street corner to a point. Run this once, then restart the kernel.

(These are also in the README's install line. If the project's `.venv` is the notebook's kernel, they install there.)

In [ ]:
%pip install osmnx scipy

  Using cached osmnx-2.1.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached scipy-1.18.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.5.2-cp314-cp314-macosx_10_15_universal2.whl.metadata (46 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
Using cached osmnx-2.1.1-py3-none-any.whl (104 kB)
Using cached scipy-1.18.1-cp314-cp314-macosx_14_0_arm64.whl (20.5 MB)
Using cached networkx-3.7-py3-none-any.whl (2.1 MB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.5.2-cp314-cp314-macosx_10_15_universal2.whl (367 kB)
Using cached idna-3.20-py3-none-any.whl (69 kB)
Using cached urllib3-2.8.0-py3-none-any.whl (135 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [osmnx]32m2/7 [networkx]
Note: you may need to

## 1) Import libraries and settings
All distances are worked out in metres, in Montréal's **MTM zone 8** projection (`EPSG:32188`, the same one as Lab 4's Recipe 2). The output is converted back to longitude and latitude at the end.

**Why a projection:** longitude and latitude are angles, not distances. At Montréal's latitude, one degree of longitude is about 78 km but one degree of latitude is about 111 km, so measuring "50 m around a route" in degrees would give a squashed oval, not a circle. MTM zone 8 is the official flat grid for the Montréal area, measured in metres, with very little distortion this close to its centre.

**The settings** control the trade-offs of the search in step 5. They are all in one cell so they're easy to find and change:
- `TRIES`: more tries = better walks, but slower. 300 × 18 walks takes about a minute.
- `TOLERANCE`: how far from the chosen length a walk can be. Too strict and few loops pass; too loose and a "3 km" walk could really be 4 km.
- `MAX_REPEAT`: stops "out and back" walks that go down a street and come straight back the same way, which are boring to walk.
- `NEAR_M`: 50 m is about half a block, close enough to see the cat or garden from the route, or reach it with a tiny detour.
- `random.seed`: the search uses random choices. Fixing the seed means the same choices every run, so the walks only change when the data or the settings change, not by chance.

In [2]:
import os
import random
import networkx as nx          # graphs and route-finding (shortest paths)
import osmnx as ox             # downloads OpenStreetMap streets as a NetworkX graph
import geopandas as gpd        # map layers (GeoJSON) as tables, with projections
from shapely.geometry import LineString, Point   # shapes: a route is a LineString, a start is a Point

MTM = "EPSG:32188"                       # metres, for measuring
GRAPH_PATH = "../data/plateau_walk.graphml"
OUT_PATH = "../data/walks.geojson"

# The three fixed starts: two métro stations and a park, all easy to reach without a car.
# These are the same three starts as in the website's walk menu.
STARTS = {                               # name: (latitude, longitude)
    "Métro Mont-Royal": (45.5240948, -73.5830385),
    "Parc Laurier": (45.5320804, -73.5878482),
    "Métro Laurier": (45.5269788, -73.5892582),
}
LENGTHS_KM = [2, 3, 5]   # a short, a medium and a long walk
TRIES = 300          # candidate loops tried for each start, length and theme
TOLERANCE = 0.15     # a "3 km" walk can be 2.55 to 3.45 km
MAX_REPEAT = 0.25    # at most a quarter of a walk can go back over streets it already used
NEAR_M = 50          # a cat, garden or fountain within 50 m of the route counts as "passed"
random.seed(464)     # same random choices every run, so the walks don't change each time
ox.settings.use_cache = False   # no extra cache folder: the network is saved as a .graphml instead

## 2) Download the walking network, or load the saved copy
`graph_from_polygon` downloads every path you can walk on inside an area: sidewalks, footpaths, park paths, pedestrian streets and **alleys**. `network_type="walk"` leaves out highways and other roads people can't walk on. The area is the Plateau-Mont-Royal borough (which includes Mile End) plus 500 m all round, so walks near the edge aren't cut off.

The network is a **graph**: street corners are *nodes* and the street pieces between them are *edges*, each with its length in metres.

**Why save it as a `.graphml` file:** downloading takes about a minute, and OpenStreetMap changes all the time (people edit it daily). Saving one copy means the notebook is fast after the first run, works offline, and gives the same walks every time. To get fresh streets, delete `data/plateau_walk.graphml` and run this cell again.

**Why only the Plateau:** that's where almost all of my cats are. A bigger area would mean a much bigger network file for the website (step 8) for walks that would pass almost no cats.

In [3]:
if os.path.exists(GRAPH_PATH):
    G = ox.load_graphml(GRAPH_PATH)
    print("Loaded the saved network from", GRAPH_PATH)
else:
    print("Downloading the walking network from OpenStreetMap (about a minute)...")
    # The borough's outline, looked up by name in OpenStreetMap's address search
    area = ox.geocode_to_gdf("Le Plateau-Mont-Royal, Montréal, Québec, Canada")
    # buffer() works in the layer's units, so go to metres (MTM) to add 500 m, then back to degrees for OSMnx
    area = area.to_crs(MTM).buffer(500).to_crs("EPSG:4326")        # add 500 m all round
    G = ox.graph_from_polygon(area.iloc[0], network_type="walk")
    ox.save_graphml(G, GRAPH_PATH)
    print("Saved to", GRAPH_PATH)

G = ox.project_graph(G, to_crs=MTM)        # coordinates in metres
# OSMnx stores each street twice, once per direction (u→v and v→u), because cars have one-way streets.
# A walker can go either way, so the two are merged into one: each street piece is counted once,
# which keeps the "doubling back" measure in step 4 simple.
G = ox.convert.to_undirected(G)            # walking works both ways on every street
print(len(G.nodes), "street corners,", len(G.edges), "street pieces")

Loaded the saved network from ../data/plateau_walk.graphml
13372 street corners, 19080 street pieces


## 3) Load the map layers and mark the green alleys on the network
A street piece counts as **green** if more than half of it lies inside a green alley (with 8 m of slack, since the alley shapes and the OpenStreetMap lines don't line up exactly).

For nature walks, the route-finder is told that green pieces are **"cheaper" to walk**: they count as 40% of their real length, so the shortest route prefers going through green alleys.

**How "green" is measured:** the city's green alleys are *areas* (polygons) and the OpenStreetMap alleys are *lines*, drawn by different people, so they don't sit exactly on top of each other. Growing each alley area by 8 m (a buffer) makes sure the OSM line falls inside it. Then, for each street piece, the length of the part inside an alley is measured. The "more than half" rule stops ordinary streets that just *cross* the end of an alley from counting as green.

**Why 40%:** the cost is `length − 0.6 × green metres`, so a fully green piece "costs" 40% of its length. That's strong enough to make routes go a block or two out of their way to use an alley, without making them zigzag wildly through every alley in the borough. Changing the 0.6 changes how keen nature walks are on alleys.

**Why "nearest corner":** routes go from corner to corner, so to steer a walk towards a cat, the notebook needs a *corner* to aim for. Each cat and garden is matched to the street corner closest to it.

In [4]:
# All four layers in metres, so they can be measured against the network
cats = gpd.read_file("../data/cat_photos.geojson").to_crs(MTM)
alleys = gpd.read_file("../data/ruelles_vertes.geojson").to_crs(MTM)
gardens = gpd.read_file("../data/communal_gardens.geojson").to_crs(MTM)
fountains = gpd.read_file("../data/water_fountains.geojson").to_crs(MTM)

# The street pieces as a table (one row per edge, with its line shape and length)
edges = ox.graph_to_gdfs(G, nodes=False)
# All the green alleys grown by 8 m and merged into one shape, so each street piece is compared with one shape
alley_area = alleys.buffer(8).union_all()
# Metres of each street piece that fall inside a green alley
inside = edges.geometry.intersection(alley_area).length
# Keep that length only if it's more than half of the piece; otherwise the piece isn't green (0)
edges["green_m"] = inside.where(inside > 0.5 * edges.geometry.length, 0)
# Copy the result from the table back onto the graph's edges, so routes can use it
nx.set_edge_attributes(G, edges["green_m"].to_dict(), "green_m")

# The "nature cost" of each piece: its length, minus 60% of its green metres
# (a fully green piece costs 40% of its length; an ordinary street costs its full length)
for u, v, k, d in G.edges(keys=True, data=True):
    d["nature_cost"] = d["length"] - 0.6 * d["green_m"]

print(round(edges["green_m"].sum() / 1000, 1), "km of the network runs through green alleys")

# The street corner nearest each cat and each garden, used to steer walks towards them
# (a set, so a corner shared by several cats is listed once)
cat_nodes = set(ox.distance.nearest_nodes(G, cats.geometry.x, cats.geometry.y))
garden_nodes = set(ox.distance.nearest_nodes(G, gardens.geometry.x, gardens.geometry.y))

23.2 km of the network runs through green alleys


## 4) Helper functions
- `edge(u, v)`: the street piece between two corners.
- `route_line(path)`: turns a list of corners into one line that follows the real street shapes.
- `make_loop(start, a, b, weight)`: a loop **start → a → b → start**, using the shortest route for each leg. Picking `a` and `b` about a third of the walk's length away from the start gives a loop of roughly the right length.
- `score(path)`: measures a loop: its length, how much of it doubles back, and the cats, green alley metres, gardens and fountains along it.

**Why helpers:** step 5 builds and measures thousands of loops (300 tries × 18 walks). Writing these jobs once as functions keeps step 5 short and readable, and means each job is done the same way every time.

**A route is a list of corners.** `nx.shortest_path` returns the corners in order, e.g. `[corner1, corner7, corner12, ...]`. To draw it, or measure what's near it, it has to become a real line that follows the street's bends, which is what `route_line` does.

**"Doubling back" (`repeat`):** each street piece is counted once in a set of *unique* pieces. If the loop's total length is 3 km but its unique pieces only add up to 2.4 km, then 0.6 km (20%) was walked twice, so `repeat` = 0.2.

In [5]:
def edge(u, v):
    # there can be more than one street piece between two corners: take the shortest
    # (e.g. a sidewalk and a park path that both join the same two corners)
    return min(G.get_edge_data(u, v).values(), key=lambda d: d["length"])


def route_line(path):
    # Start with the first corner's position, then add each street piece's shape in turn
    coords = [(G.nodes[path[0]]["x"], G.nodes[path[0]]["y"])]
    for u, v in zip(path[:-1], path[1:]):      # each pair of consecutive corners = one street piece
        data = edge(u, v)
        if "geometry" in data:
            # A curved or bent street: OSMnx stores its full shape
            piece = list(data["geometry"].coords)
            # The stored shape may be drawn from v to u; if so, flip it so the line doesn't jump back and forth
            if piece[0] != (G.nodes[u]["x"], G.nodes[u]["y"]):
                piece = piece[::-1]          # make the piece run in the walking direction
        else:
            # A straight street: OSMnx stores no shape, so it's just the two corners
            piece = [(G.nodes[u]["x"], G.nodes[u]["y"]), (G.nodes[v]["x"], G.nodes[v]["y"])]
        coords.extend(piece[1:])   # [1:]: skip the first point, it is already the end of the previous piece
    return LineString(coords)


def make_loop(start, a, b, weight):
    # weight = "length" (shortest in metres) or "nature_cost" (shortest, with green alleys cheaper)
    try:
        leg1 = nx.shortest_path(G, start, a, weight=weight)
        leg2 = nx.shortest_path(G, a, b, weight=weight)
        leg3 = nx.shortest_path(G, b, start, weight=weight)
    except nx.NetworkXNoPath:
        # Some corners can't be reached (e.g. a path cut off from the rest): skip this try
        return None
    # Join the three legs. Each leg starts where the last one ended, so [1:] drops that repeated corner.
    return leg1 + leg2[1:] + leg3[1:]


def score(path):
    steps = list(zip(path[:-1], path[1:]))                            # every street piece walked, in order
    length = sum(edge(u, v)["length"] for u, v in steps)              # total metres walked
    # The same street can be walked as (u, v) or (v, u); sorting the pair makes both the same,
    # so the set holds each street piece only once (key=str: corner ids can be mixed types)
    unique = {tuple(sorted(step, key=str)) for step in steps}     # each street piece once
    line = route_line(path)
    near = line.buffer(NEAR_M)        # a 50 m wide band around the route: anything inside it counts as passed
    return {
        "geometry": line,
        "length_m": length,
        # share of the length that is a second walk down a street already used (0 = none, 0.25 = a quarter)
        "repeat": 1 - sum(edge(u, v)["length"] for u, v in unique) / length,
        "cats": int(cats.within(near).sum()),           # cats inside the 50 m band
        "green_m": sum(edge(u, v)["green_m"] for u, v in unique),   # green alley metres, each piece once
        "gardens": int(gardens.within(near).sum()),
        "fountains": int(fountains.within(near).sum()),
    }

## 5) Try many loops, keep the best one for each start, length and theme
For each start and length:
1. Find every corner whose walking distance from the start is about a third of the walk's length (the *ring*).
2. Try 300 loops through two corners from the ring. **Cat walks** usually pick corners next to cats; **nature walks** often pick one next to a garden, and route through green alleys.
3. Throw away loops that are too short, too long, or double back too much.
4. Keep the best: **cat walks** by number of cats (green alley metres break ties); **nature walks** by green alley metres, plus 300 m for each garden passed.

**This is the heart of the walks, and the part `app.js` copies.**

**The ring (22–38% of the length).** "A third" is the idea; the range around it gives plenty of corners to choose from. The distances are real *walking* distances along the streets (not straight lines), found with one Dijkstra search from the start. The `cutoff` stops that search at half the walk's length, since no corner of a loop can be further away than that, which saves a lot of time.

**The wider band for cats and gardens (15–45%).** Only some ring corners are next to a cat or a garden. Widening the band for those gives the search more of them to aim at.

**Aim, but also explore.** Cat walks aim at two cat corners 70% of the time and use two random ring corners the other 30%. Nature walks aim at a garden 50% of the time. The random tries matter: a loop between two random corners sometimes passes *more* cats on the way than one aimed at two cats, and without them the search would never find it.

**The score:**
- **Cat walk:** `cats × 100 + green metres ÷ 10`. Cats count most. One cat is worth as much as 1 km of green alley, so between two loops with the same number of cats, the greener one wins.
- **Nature walk:** `green metres + 300 × gardens`. Each community garden passed is worth as much as 300 m of green alley.

The printout shows the winner for each combination, so you can see the trade-off: cat walks pass many more cats, nature walks many more metres of green alley.

In [6]:
rows = []   # one row per finished walk; becomes the GeoJSON in step 6
for name, (lat, lon) in STARTS.items():
    # The start in metres (MTM), then snapped to the nearest street corner, since routes go corner to corner
    x, y = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326").to_crs(MTM).iloc[0].coords[0]
    start = ox.distance.nearest_nodes(G, x, y)

    for km in LENGTHS_KM:
        target = km * 1000   # the walk's length in metres
        # Walking distance from the start to every corner up to half the target away (one Dijkstra search)
        dist = nx.single_source_dijkstra_path_length(G, start, cutoff=target / 2, weight="length")
        # The ring: corners about a third of the walk away, so start → a → b → start is about the right length
        ring = [n for n, d in dist.items() if 0.22 * target <= d <= 0.38 * target]
        # A wider band, to find more cats and gardens to aim at
        wide = [n for n, d in dist.items() if 0.15 * target <= d <= 0.45 * target]
        ring_cats = [n for n in wide if n in cat_nodes]          # corners in the band next to a cat
        ring_gardens = [n for n in wide if n in garden_nodes]    # corners in the band next to a garden

        for theme in ["cats", "nature"]:
            best, best_value = None, -1   # the best loop so far, and its score
            for _ in range(TRIES):
                # Choose the two corners the loop goes through:
                if theme == "cats" and len(ring_cats) >= 2 and random.random() < 0.7:
                    a, b = random.sample(ring_cats, 2)                  # 70%: two cat corners
                elif theme == "nature" and ring_gardens and random.random() < 0.5:
                    a, b = random.choice(ring_gardens), random.choice(ring)   # 50%: a garden + a ring corner
                else:
                    a, b = random.sample(ring, 2)                       # otherwise: two random ring corners
                # Cat walks take the truly shortest streets; nature walks use the cost where green alleys are cheaper
                path = make_loop(start, a, b, "length" if theme == "cats" else "nature_cost")
                if path is None:
                    continue
                s = score(path)
                # Throw away loops that are too far from the target length, or that double back too much
                if abs(s["length_m"] - target) > TOLERANCE * target or s["repeat"] > MAX_REPEAT:
                    continue
                # Score the loop (see the explanation above) and keep it if it's the best so far
                if theme == "cats":
                    value = s["cats"] * 100 + s["green_m"] / 10
                else:
                    value = s["green_m"] + 300 * s["gardens"]
                if value > best_value:
                    best, best_value = s, value

            # Save the winner with its start, length and theme (**best adds its numbers and shape).
            # Note: if no try passed the filters, best is still None and this line fails; it never
            # happened for these three starts, but a start far from the cats could cause it.
            rows.append({"start": name, "km": km, "theme": theme, **best})
            print(f"{name:17} {km} km {theme:6}  {best['length_m'] / 1000:.2f} km, {best['cats']:2} cats, "
                  f"{best['green_m']:5.0f} m green alleys, {best['gardens']} gardens, {best['fountains']} fountains")

Métro Mont-Royal  2 km cats    2.06 km, 21 cats,     0 m green alleys, 0 gardens, 2 fountains
Métro Mont-Royal  2 km nature  2.26 km,  4 cats,   912 m green alleys, 0 gardens, 1 fountains
Métro Mont-Royal  3 km cats    2.95 km, 24 cats,     7 m green alleys, 0 gardens, 2 fountains
Métro Mont-Royal  3 km nature  3.13 km,  1 cats,  1549 m green alleys, 0 gardens, 1 fountains
Métro Mont-Royal  5 km cats    4.92 km, 27 cats,     6 m green alleys, 0 gardens, 5 fountains
Métro Mont-Royal  5 km nature  5.09 km,  1 cats,  2071 m green alleys, 1 gardens, 1 fountains
Parc Laurier      2 km cats    1.77 km,  1 cats,   292 m green alleys, 0 gardens, 2 fountains
Parc Laurier      2 km nature  2.14 km,  0 cats,   938 m green alleys, 0 gardens, 2 fountains
Parc Laurier      3 km cats    3.14 km,  7 cats,   203 m green alleys, 0 gardens, 3 fountains
Parc Laurier      3 km nature  3.21 km,  1 cats,  1540 m green alleys, 0 gardens, 2 fountains
Parc Laurier      5 km cats    4.84 km, 27 cats,     6 m gre

## 6) Write the walks
One line per walk, with what the website's walk panel shows. **Output:** `../data/walks.geojson`

The walks are turned back into longitude and latitude (`EPSG:4326`), because that's what web maps (Leaflet) expect. Coordinates are rounded to 6 decimals (about 10 cm), which keeps the file small without changing anything you could see on the map.

*(The website no longer reads this file; see the top of the notebook. It's still a handy way to look at the Python walks, e.g. by dragging it into geojson.io or QGIS.)*

In [7]:
walks = gpd.GeoDataFrame(rows, geometry="geometry", crs=MTM)   # the winners as a map layer, still in metres
walks["length_km"] = (walks["length_m"] / 1000).round(2)       # 3087.4 m → 3.09 km, easier to read in a popup
walks["green_m"] = walks["green_m"].round(0).astype(int)       # whole metres
# Only the columns the walk panel needs, in a sensible order (drops length_m and repeat)
walks = walks[["start", "km", "theme", "length_km", "cats", "green_m", "gardens", "fountains", "geometry"]]

walks.to_crs("EPSG:4326").to_file(OUT_PATH, driver="GeoJSON", COORDINATE_PRECISION=6)
print("Wrote", OUT_PATH, "-", round(os.path.getsize(OUT_PATH) / 1_000_000, 2), "MB")
walks.drop(columns="geometry")   # show the table without the long line shapes

Wrote ../data/walks.geojson - 0.14 MB


,start,km,theme,length_km,cats,green_m,gardens,fountains
0,Métro Mont-Royal,2,cats,2.06,21,0,0,2
1,Métro Mont-Royal,2,nature,2.26,4,912,0,1
2,Métro Mont-Royal,3,cats,2.95,24,7,0,2
3,Métro Mont-Royal,3,nature,3.13,1,1549,0,1
4,Métro Mont-Royal,5,cats,4.92,27,6,0,5
5,Métro Mont-Royal,5,nature,5.09,1,2071,1,1
6,Parc Laurier,2,cats,1.77,1,292,0,2
7,Parc Laurier,2,nature,2.14,0,938,0,2
8,Parc Laurier,3,cats,3.14,7,203,0,3
9,Parc Laurier,3,nature,3.21,1,1540,0,2


## 7) Check the file
The check cell from Lab 4, with the four settings changed for this file. 3 starts x 3 lengths x 2 themes = 18 walks.

It reads the file back *from disk* (not the table still in memory), so it checks what the website would actually get: the right number of walks, in longitude and latitude, inside a box around the Plateau, with every field the popup needs, and small enough to load quickly.

In [8]:
# Reads the saved file back and checks it, like the Lab 4 check cell
# ---- settings: change these for your own data ----
FILE = OUT_PATH
WEST, EAST, SOUTH, NORTH = -73.63, -73.54, 45.50, 45.55   # a box around the Plateau, in degrees
EXPECTED = 18             # how many features you expect (rows after your filters)
POPUP_FIELDS = ["start", "km", "theme", "length_km", "cats", "green_m", "gardens", "fountains"]
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['LineString']
All checks passed


## 8) Export the walking network for the website ("My own spot" walks)
The website can also plan a loop from **any spot the reader chooses** (an address or a tap on the map). Python doesn't run on the website, so the page does the route-finding itself in `app.js` (section 11), with the same method as step 5. For that it needs the walking network, but `plateau_walk.graphml` is 16 MB, far too big for a web page.

This step writes a small copy with only what the page needs:
- **corners**: longitude and latitude of each street corner;
- **street pieces**: the two corners they join, their length and metres of green alley (from step 3), and the bends in between (simplified to 2 m).

**Output:** `../data/walk_network.json` (under 1 MB).

**Today this is the step the website depends on**: since the "Stops to include" update, *every* walk on the map is planned from this file. Re-run it whenever the network (step 2) or the green alleys (step 3) change.

**How it gets from 16 MB to under 1 MB:**
- **Only the needed facts.** The `.graphml` keeps every OpenStreetMap tag (street names, surface, lighting, ...). The page only needs positions, lengths and green metres.
- **Short corner numbers.** OpenStreetMap ids are long numbers like `213827519`. Here corners are renumbered 0, 1, 2, ..., and street pieces refer to those small numbers.
- **One piece per pair of corners.** Where two pieces join the same two corners, only the shortest is kept (as `edge()` does in step 4), since a route would always take it anyway.
- **Rounding and simplifying.** 5 decimals of longitude/latitude is about 1 m. Bends closer than 2 m to a straight line are dropped (`simplify(2)`), which removes many points without changing the drawn route visibly.
- **No spaces.** `separators=(",", ":")` writes the JSON without any spaces.

**The file's layout** (lists, not named fields, to keep it small):
```
{"nodes": [[lon, lat], ...],                                  corner 0, corner 1, ...
 "edges": [[from, to, length_m, green_m, [lon, lat, lon, lat, ...]], ...]}   the bends, flattened
```
`app.js` reads this into its own graph and runs its own Dijkstra on it.

In [ ]:
import json

NETWORK_PATH = "../data/walk_network.json"

# One street piece per pair of corners (the shortest, as edge() does), skipping pieces that loop back to the same corner
web = edges.reset_index()                       # u and v (the two corners) become ordinary columns
web = web[web["u"] != web["v"]]                 # drop little loops that start and end at the same corner
web["pair"] = [tuple(sorted((u, v), key=str)) for u, v in zip(web["u"], web["v"])]   # (u, v) and (v, u) → same pair
web = web.sort_values("length").drop_duplicates("pair")   # shortest first, so drop_duplicates keeps the shortest

# Corners: number them 0, 1, 2... and keep their longitude and latitude
corners = ox.graph_to_gdfs(G, edges=False).to_crs("EPSG:4326")
used = sorted(set(web["u"]) | set(web["v"]), key=str)     # only corners that some street piece uses
number = {n: i for i, n in enumerate(used)}               # OpenStreetMap id → short number
nodes_out = [[round(corners.geometry[n].x, 5), round(corners.geometry[n].y, 5)] for n in used]   # 5 decimals ≈ 1 m

# Street pieces: simplify the shape (2 m), turn it to longitude and latitude, and make it run from u to v
web["geometry"] = web.geometry.simplify(2)     # done in metres (still MTM here), so "2" means 2 m
web = web.set_geometry("geometry").to_crs("EPSG:4326")
edges_out = []
for u, v, length, green, line in zip(web["u"], web["v"], web["length"], web["green_m"], web.geometry):
    coords = list(line.coords)
    ux, uy = nodes_out[number[u]]
    # If the line's LAST point is closer to corner u than its first point, the line is drawn from v to u: flip it.
    # (Squared distances are enough to compare which is closer, so no square root is needed.)
    if (coords[0][0] - ux) ** 2 + (coords[0][1] - uy) ** 2 > (coords[-1][0] - ux) ** 2 + (coords[-1][1] - uy) ** 2:
        coords = coords[::-1]
    middle = [round(c, 5) for x, y in coords[1:-1] for c in (x, y)]   # the bends, without the two corners
    # [from corner, to corner, length in m, green alley m, bends as one flat list]
    edges_out.append([number[u], number[v], round(float(length), 1), round(float(green), 1), middle])

with open(NETWORK_PATH, "w") as f:
    json.dump({"nodes": nodes_out, "edges": edges_out}, f, separators=(",", ":"))   # no spaces, smallest file
print(len(nodes_out), "corners,", len(edges_out), "street pieces")
print("Wrote", NETWORK_PATH, "-", round(os.path.getsize(NETWORK_PATH) / 1_000_000, 2), "MB")

## 9) Next
- **The website's walks** come from `data/walk_network.json` (step 8). `app.js` plans each loop in the reader's browser with the ideas from step 5: a ring of corners about a third of the walk away, one corner next to each ticked kind of stop, many random tries, a filter for length and doubling back, and a score. Its settings (`WALK_TOLERANCE`, `WALK_MAX_REPEAT`, `WALK_NEAR_M`, the 0.6 green discount) match the ones here.
- **To change the area** (e.g. if cats from other boroughs are added): change the place name in step 2, delete `data/plateau_walk.graphml`, and run everything again. Then check the new `walk_network.json` is still a reasonable size for a web page (under about 5 MB).
- **When the city updates the green alleys** (every autumn): run steps 1, 2, 3 and 8 again, so the green metres in `walk_network.json` are up to date.
- `data/walks.geojson` (steps 4–7) isn't used by the site any more. To change those walks anyway, edit `STARTS` and `LENGTHS_KM` in step 1 and run everything again; the saved network is reused, so it only takes about a minute.